In [ ]:


# Exercise
"""EXERCISE: Build a chatbot with:
1. Persistent memory (SQLite)
2. Automatic summarization after 10 messages
3. User preference tracking
Hint: Combine RunnableWithMessageHistory with SQLChatMessageHistory
"""
   
import os
import sqlite3

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory


def exercise_persistent_memory():

    # ============================================================
    # 1. DATABASE SETUP
    # ============================================================

    db_path = "./chat_history.db"
    connection_string = f"sqlite:///{db_path}"
    session_id = "persistent_user"

    # Clean slate: remove old database if it exists
    if os.path.exists(db_path):
        os.remove(db_path)

    # Configuration used by RunnableWithMessageHistory
    config = {
        "configurable": {
            "session_id": session_id
        }
    }

    # ============================================================
    # 2. FUNCTION TO BUILD A NEW CHATBOT CHAIN
    # ============================================================

    def build_chain():

        # Create the LLM
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.7
        )

        # Function that retrieves chat history from SQLite
        def get_session_history(
            session_id: str
        ) -> BaseChatMessageHistory:

            return SQLChatMessageHistory(
                session_id=session_id,
                connection=connection_string
            )

        # Create the chat prompt
        prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                "You are a helpful assistant. Remember user preferences."
            ),

            # Previous conversation will be inserted here
            MessagesPlaceholder(
                variable_name="history"
            ),

            # Current user message
            (
                "human",
                "{input}"
            )
        ])

        # Basic chain:
        # Prompt → LLM → String output
        chain = prompt | llm | StrOutputParser()

        # Add persistent message history
        chain_with_history = RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="input",
            history_messages_key="history"
        )

        return chain_with_history

    # ============================================================
    # 3. RUN 1 — STORE USER PREFERENCES
    # ============================================================

    print("\n" + "=" * 60)
    print("RUN 1 — STORING USER PREFERENCES")
    print("=" * 60)

    chain_v1 = build_chain()

    run1_messages = [
        "My name is Anushka. I prefer light mode themes and Python over Java.",
        "I also like my responses concise — no fluff."
    ]

    for msg in run1_messages:

        print(f"\nUser: {msg}")

        response = chain_v1.invoke(
            {"input": msg},
            config=config
        )

        print(f"AI: {response}")

    # Destroy the first chain
    # This demonstrates that the chain itself is not responsible
    # for permanently storing the conversation.
    del chain_v1

    # ============================================================
    # 4. DATABASE CHECK AFTER RUN 1
    # ============================================================

    print("\n" + "=" * 60)
    print("DATABASE CHECK AFTER RUN 1")
    print("=" * 60)

    print(f"Database file exists: {os.path.exists(db_path)}")

    if os.path.exists(db_path):
        print(
            f"Database size: "
            f"{os.path.getsize(db_path)} bytes"
        )

    conn = sqlite3.connect(db_path)

    cursor = conn.execute(
        "SELECT * FROM message_store ORDER BY rowid"
    )

    rows = cursor.fetchall()

    print(f"\nTotal messages stored: {len(rows)}")

    for i, row in enumerate(rows):

        print(
            f"Row {i + 1}: "
            f"session={row[0] if len(row) > 0 else 'N/A'}, "
            f"message={str(row[1])[:80] if len(row) > 1 else 'N/A'}"
        )

    conn.close()

    # ============================================================
    # 5. RUN 2 — NEW CHAIN, SAME DATABASE
    # ============================================================

    print("\n" + "=" * 60)
    print("RUN 2 — NEW CHAIN, TESTING PERSISTENT MEMORY")
    print("=" * 60)

    # This is a completely new chain
    chain_v2 = build_chain()

    recall_questions = [
        "What is my name?",
        "What theme do I prefer?",
        "What programming language do I prefer?",
        "How do I like my responses?"
    ]

    for msg in recall_questions:

        print(f"\nUser: {msg}")

        response = chain_v2.invoke(
            {"input": msg},
            config=config
        )

        print(f"AI: {response}")

    # Destroy the second chain
    del chain_v2

    # ============================================================
    # 6. FINAL DATABASE CHECK
    # ============================================================

    print("\n" + "=" * 60)
    print("FINAL DATABASE STATE")
    print("=" * 60)

    conn = sqlite3.connect(db_path)

    cursor = conn.execute(
        "SELECT COUNT(*) FROM message_store"
    )

    count = cursor.fetchone()[0]

    conn.close()

    print(
        f"Total messages in DB after both runs: {count}"
    )

    print("\nKey insight:")
    print("Run 2 used a completely new chain.")
    print("The conversation was recovered from SQLite.")
    print("Therefore, the memory persisted outside the chain.")

    # ============================================================
    # 7. CLEANUP
    # ============================================================

    if os.path.exists(db_path):
        os.remove(db_path)

    print("\nDatabase cleaned up.")


# ================================================================
# RUN THE EXERCISE
# ================================================================

exercise_persistent_memory()

